# 03 — Evaluation and Demo

Metrics (blueprint section 9): execution success, result match
(canonicalized, position-based), first-attempt success, repair success,
unsafe block rate, and median latency. **No LLM judge** — everything is
computed from executed results.

`gold-replay` validates the harness itself without a model; run with
`--backend transformers` on a GPU (or `--backend ollama` locally) for real
model quality numbers.


In [ ]:
# Cell 00 — environment check + repo bootstrap (works on Colab and locally)
import importlib.util
import platform
import sqlite3
import subprocess
import sys
from pathlib import Path

print("python :", platform.python_version())
print("sqlite :", sqlite3.sqlite_version)

try:
    import torch
    print("torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU    :", torch.cuda.get_device_name(0))
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    print("torch  : not installed (CPU-only mode — scripted/ollama backends still work)")
    HAS_GPU = False

# Locate the repo root whether we run from notebooks/, the repo root, or Colab.
def find_repo_root() -> Path:
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / "src" / "sql_agent").is_dir():
            return base
    # Colab: clone if the repo is not present yet (replace URL with your fork).
    clone_dir = Path.cwd() / "local-enterprise-sql-agent"
    if not clone_dir.is_dir():
        raise RuntimeError(
            "repo not found — on Colab run: "
            "!git clone <YOUR_REPO_URL> local-enterprise-sql-agent"
        )
    return clone_dir

REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
print("repo   :", REPO_ROOT)

if importlib.util.find_spec("sql_agent") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)], check=True)
import sql_agent
print("package:", sql_agent.__version__)


In [ ]:
import subprocess, sys

backend_choice = 'transformers' if HAS_GPU else 'gold-replay'
print('eval backend:', backend_choice)
proc = subprocess.run(
    [sys.executable, str(REPO_ROOT / 'scripts' / 'run_eval.py'), '--backend', backend_choice],
    capture_output=True, text=True, cwd=REPO_ROOT,
)
print(proc.stdout[-3000:])
if proc.returncode not in (0,):
    print('exit code:', proc.returncode)

In [ ]:
import json
import pandas as pd

results = pd.read_csv(REPO_ROOT / 'reports/evaluation.csv')
metrics = json.loads((REPO_ROOT / 'reports/metrics.json').read_text())
display(results[['question_id', 'category', 'status', 'attempts',
                 'first_attempt_executed', 'final_executed', 'result_match',
                 'blocked', 'latency_ms']])
print(json.dumps(metrics, indent=2))

## Attempt traces
Every run stores its full state machine trace under `reports/traces/`.

In [ ]:
trace = json.loads((REPO_ROOT / 'reports/traces/biz_004.json').read_text())
print('question :', trace['question'])
print('status   :', trace['status'])
for attempt in trace['attempts']:
    sql = (attempt.get('candidate') or {}).get('sql', '')
    print(f"attempt {attempt['attempt_no']}: {sql[:100]}")

## Chart demo — rendered only from executed rows (never invented values)

In [ ]:
from sql_agent import ChartHint, ReadOnlyExecutor, maybe_chart
from sql_agent.schemas import AgentConfig

executor = ReadOnlyExecutor(REPO_ROOT / 'data/databases/retail.sqlite', AgentConfig())
execution = executor.execute(
    "SELECT strftime('%Y-%m', o.order_date) AS month, "
    "ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue "
    "FROM orders AS o JOIN order_items AS oi ON o.order_id = oi.order_id "
    "WHERE o.status = 'completed' AND o.order_date >= '2025-01-01' "
    "AND o.order_date < '2026-01-01' GROUP BY month ORDER BY month"
)
path = maybe_chart(execution, ChartHint(kind='line', x='month', y=['revenue']),
                   REPO_ROOT / 'assets/eval_revenue_2025.png')
from IPython.display import Image, display
display(Image(str(path)))

## Reading the numbers honestly
- `gold-replay` proves the harness works; its business metrics are by
  construction, only the **safety block rate is a real measurement**.
- Real model metrics belong in README with sample counts and the exact
  model + prompt version (see `provenance` in each trace).
- Investigate failures in `reports/failure_analysis.md` — wrong table,
  wrong join, missing aggregation, column hallucination, unsafe attempt,
  empty result.
